# 04 — Multimodal fusion + `MultimodalSCDModel`

End-to-end sanity check: synthetic cohort → `StandardPreprocessor` → batch tensors
→ `MultimodalSCDModel` with each fusion strategy (`attention`, `cross`, `late`).


In [ ]:
import sys
from pathlib import Path

_repo = Path.cwd()
for _ in range(4):
    if (_repo / "src" / "mmvlm4scd").is_dir():
        sys.path.insert(0, str(_repo / "src"))
        break
    _repo = _repo.parent

import torch
from torch.utils.data import DataLoader

from mmvlm4scd.data import MultimodalSCDDataset, StandardPreprocessor, generate_synthetic_cohort
from mmvlm4scd.data.synthetic import SCDSyntheticConfig
from mmvlm4scd.models import ModelConfig, MultimodalSCDModel


In [ ]:
cohort = generate_synthetic_cohort(SCDSyntheticConfig(n_patients=128, seed=0))
pre = StandardPreprocessor().fit(cohort["clinical"])
x_clin = pre.transform(cohort["clinical"])

ds = MultimodalSCDDataset(
    clinical=x_clin,
    genomic=cohort["genomic"],
    imaging=cohort["imaging"],
    temporal=cohort["temporal"],
    severity=cohort["severity"],
    survival_time=cohort["survival_time"],
    survival_event=cohort["survival_event"],
)
loader = DataLoader(ds, batch_size=32, shuffle=False, drop_last=False)
batch = next(iter(loader))
{k: v.shape for k, v in batch.items()}


In [ ]:
def build_model(fusion: str):
    cfg = ModelConfig(
        clinical_input_dim=x_clin.shape[1],
        genomic_input_dim=cohort["genomic"].shape[1],
        imaging_input_dim=cohort["imaging"].shape[1],
        temporal_input_dim=cohort["temporal"].shape[2],
        embed_dim=64,
        fusion=fusion,
    )
    return MultimodalSCDModel(cfg)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

for fusion in ("attention", "cross", "late"):
    model = build_model(fusion).to(device)
    b = {k: v.to(device) for k, v in batch.items()}
    out = model(b)
    print(fusion, "| embedding", tuple(out["embedding"].shape),
          "| severity_logits", tuple(out["severity_logits"].shape),
          "| risk_score", tuple(out["risk_score"].shape))


## Optional: one optimizer step (same stack as unit tests)

Demonstrates that gradients flow through all fusion modes.


In [ ]:
import torch.nn.functional as F

from mmvlm4scd.training.losses import cox_partial_likelihood_loss


def one_step(fusion: str):
    torch.manual_seed(0)
    model = build_model(fusion).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    b = {k: v.to(device) for k, v in batch.items()}
    opt.zero_grad(set_to_none=True)
    out = model(b)
    loss_cls = F.cross_entropy(out["severity_logits"], b["severity"])
    loss_cox = cox_partial_likelihood_loss(
        out["risk_score"], b["survival_time"], b["survival_event"]
    )
    (loss_cls + 0.1 * loss_cox).backward()
    opt.step()
    return float(loss_cls.detach()), float(loss_cox.detach())

for fusion in ("attention", "cross", "late"):
    lc, lx = one_step(fusion)
    print(f"{fusion}: CE={lc:.4f} Cox={lx:.4f}")
